In [ ]:
# 1. 필요한 라이브러리 설치
!pip install openai==0.28 transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 968.8 kB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.76.2
    Uninstalling openai-1.76.2:
      Successfully uninstalled openai-1.76.2


In [ ]:
# 2. 라이브러리 불러오기
import openai
import random
import json
import time
from transformers import T5Tokenizer, T5ForConditionalGeneration, pipeline
from sentence_transformers import SentenceTransformer, util
from google.colab import files

In [ ]:
# 3. OpenAI API 키 설정  ---->  깃허브 올릴 때 가리기
openai.api_key = "***"

In [ ]:
# 4. JSON 데이터 업로드
uploaded = files.upload()
with open("open.json", "r", encoding="utf-8") as f:
    data = json.load(f)

Saving open_problem.json to open_problem.json


In [ ]:
# 5. GPT 번역 함수
def gpt_translate(text, source_lang="Korean", target_lang="English"):
    prompt = f"Translate the following text from {source_lang} to {target_lang}:\n\n{text}"

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful translation assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=512
        )
        translated = response['choices'][0]['message']['content'].strip()
        return translated
    except Exception as e:
        print(f"[ERROR] GPT translation failed: {e}")
        return None

In [ ]:
# 6. T5 패러프레이즈 모델 로딩
model_name = "ramsrigouthamg/t5_paraphraser"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
paraphraser = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

def generate_paraphrases(text, num_return_sequences=3):
    input_text = f"paraphrase: {text}"
    outputs = paraphraser(
        input_text,
        max_length=100,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=100,
        top_p=0.92,
        temperature=0.8
    )
    return [output['generated_text'] for output in outputs]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# 7. 동의어 사전 및 치환 함수
synonym_dict = {
    "구하시오": ["계산하시오", "알아보시오", "찾으시오", "답을 구하시오"],
    "몇 개일까": ["몇 개일까", "몇 개인가요", "몇 개인가", "총 몇 개인가요"],
    "가능한 경우의 수": ["모든 조합의 수", "경우의 수", "가능한 조합 수"],
    "이동한 거리": ["간 거리", "이동 거리", "총 이동 거리"],
    "성립시키기 위해": ["맞게 만들기 위해", "성립되도록 하기 위해", "성립하도록 하기 위해"],
    "무사히": ["안전하게", "잡히지 않고", "성공적으로"],
    "정답은": ["답은", "해답은", "결과는"],
    "바로": ["즉시", "곧바로", "즉각적으로"],
    "빠진 숫자": ["누락된 숫자", "말하지 않은 숫자", "빠뜨린 숫자"],
    "중량 측정 저울": ["전자저울", "무게 측정기", "저울"],
    "총 걸린 시간": ["전체 소요 시간", "총 소요 시간", "걸린 전체 시간"],
    "펀치를 이용해서": ["구멍 뚫는 기구로", "펀치기를 써서", "펀치기로"],
    "보안카드": ["출입카드", "실험실 카드", "보안 문자열 카드"],
    "문자열": ["숫자열", "비밀번호", "문자코드"],
    "동시에": ["같은 시점에", "한꺼번에", "한 시점에"],
}

def synonym_replace(korean_text):
    if not korean_text:
        return ""
    words = korean_text.split()
    new_words = []
    for word in words:
        for key, synonyms in synonym_dict.items():
            if key in word and random.random() < 0.5:
                word = word.replace(key, random.choice(synonyms))
                break
        new_words.append(word)
    return ' '.join(new_words)

In [ ]:
# 8. 스타일 변환 함수
style_templates = [
        "다음을 생각해보시오: {문장}",
        "지문을 잘 읽고 본인의 생각을 적어주세요: {문장}",
        "{문장} 이 문제를 해결해 보세요.",
        "질문 드립니다! {문장}",
        "생각해 보세요: {문장}",
    ]

def style_transfer(text, apply_prob=0.6):
    if random.random() > apply_prob:
        return text

    template = random.choice(style_templates)
    styled_text = template.replace("{문장}", text.strip())

    return styled_text

In [ ]:
# 9. 의미 보존 함수
embed_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

def filter_meaning_preserved(original, candidates, threshold):
    original_emb = embed_model.encode(original, convert_to_tensor=True)
    results = []
    for c in candidates:
        c_emb = embed_model.encode(c, convert_to_tensor=True)
        sim = util.pytorch_cos_sim(original_emb, c_emb).item()
        if sim >= threshold:
            results.append(c)
    return results

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/336k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/967k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# 10. 유사문장 제거
def deduplicate_by_similarity(sentences, threshold):
    unique = []
    embeddings = embed_model.encode(sentences, convert_to_tensor=True)

    for i, sent in enumerate(sentences):
        is_duplicate = False
        for u_idx in range(len(unique)):
            sim = util.pytorch_cos_sim(embeddings[i], embeddings[sentences.index(unique[u_idx])]).item()
            if sim >= threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            unique.append(sent)
    return unique

In [ ]:
# 11. 중복 제거된 문장 찾기
def is_similar_to_any(candidate, existing_list, threshold):
    candidate_embed = embed_model.encode(candidate, convert_to_tensor=True)
    for text in existing_list:
        sim = util.pytorch_cos_sim(candidate_embed, embed_model.encode(text, convert_to_tensor=True)).item()
        if sim >= threshold:
            return True
    return False

In [ ]:
# 12. 의미 보존 + 중복 제거
def enforce_min_questions(original, candidates, min_count):
    preserved = filter_meaning_preserved(original, candidates, threshold=0.75)
    deduped = deduplicate_by_similarity(preserved, threshold=0.92)

    if len(deduped) < min_count:
        rejected = [p for p in preserved if p not in deduped and not is_similar_to_any(p, deduped, threshold=0.92)]
        rejected_sorted = sorted(
            rejected,
            key=lambda x: -util.pytorch_cos_sim(
                embed_model.encode(original, convert_to_tensor=True),
                embed_model.encode(x, convert_to_tensor=True)
            ).item()
        )

        for r in rejected_sorted:
            deduped.append(r)
            if len(deduped) >= min_count:
                break

    return deduped

In [ ]:
# 13. 전체 백번역 증강 파이프라인
def full_back_translate_with_paraphrase_style(korean_text, num_augments):
    en_text = gpt_translate(korean_text, source_lang='Korean', target_lang='English')
    if not en_text:
        return [korean_text]

    paraphrases = generate_paraphrases(en_text, num_return_sequences=num_augments)

    results = []
    for p in paraphrases:
        ko_text = gpt_translate(p, source_lang='English', target_lang='Korean')
        if not ko_text:
            ko_text = korean_text
        ko_text_synonym = synonym_replace(ko_text)
        final_text = style_transfer(ko_text_synonym)
        results.append(final_text)

    return results

In [ ]:
# 14. 증강 실행 및 저장
augmented_data = []
for item in data:
    question = item['Question']

    augmented_questions = full_back_translate_with_paraphrase_style(question, num_augments=5)
    all_questions = [question] + augmented_questions
    final_questions = enforce_min_questions(question, all_questions, 3)

    for aug_q in final_questions:
        augmented_data.append({
            "Question": aug_q
        })

In [ ]:
# 15. JSON 저장 및 다운로드
output_filename = "augmented_open.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, ensure_ascii=False, indent=4)

files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>